## 1. Imports

In [2]:
# Cell 1
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
from collections import defaultdict
from scipy.stats import rankdata

# Import your local parsing file
import parsing 

# Scikit-learn
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Survival Analysis
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from sksurv.metrics import concordance_index_ipcw
from sksurv.util import Surv

print("Libraries imported and parsing module loaded.")

Libraries imported and parsing module loaded.


## 2. Loading data

In [3]:
# Cell 2
# Load Datasets (Adjust paths if necessary)
df = pd.read_csv("./data/X_train/clinical_train.csv")
df_eval = pd.read_csv("./data/X_test/clinical_test.csv")

maf_df = pd.read_csv("./data/X_train/molecular_train.csv")
maf_eval = pd.read_csv("./data/X_test/molecular_test.csv")

target_df = pd.read_csv("./data/target_train.csv")

# Clean Target
target_df.dropna(subset=['OS_YEARS', 'OS_STATUS'], inplace=True)
target_df['OS_YEARS'] = pd.to_numeric(target_df['OS_YEARS'], errors='coerce')
target_df['OS_STATUS'] = target_df['OS_STATUS'].astype(bool)

# Merge target into training data
df = df.merge(target_df[['ID', 'OS_YEARS', 'OS_STATUS']], on='ID', how='inner')

print(f"Training Data: {df.shape}")
print(f"Molecular Data: {maf_df.shape}")

Training Data: (3173, 11)
Molecular Data: (10935, 11)


## 3. Feature engineering

In [4]:
# Cell 3
total_features = []

# --- 1. Cytogenetics ---
print("Processing Cytogenetics...")
# Calculate Nmut (Mutation Count)
nmut_train = maf_df.groupby('ID').size().reset_index(name='Nmut')
nmut_test = maf_eval.groupby('ID').size().reset_index(name='Nmut')

df = df.merge(nmut_train, on='ID', how='left').fillna({'Nmut': 0})
df_eval = df_eval.merge(nmut_test, on='ID', how='left').fillna({'Nmut': 0})
total_features.append('Nmut')

# Parse ISCN using parsing.py
def process_cyto(dataset):
    # calls parsing.parse_CYTO
    cyto_dicts = dataset['CYTOGENETICS'].apply(parsing.parse_CYTO)
    cyto_df = pd.DataFrame(cyto_dicts.tolist(), index=dataset.index).fillna(0)
    return pd.concat([dataset, cyto_df], axis=1)

df = process_cyto(df)
df_eval = process_cyto(df_eval)

# Add valid cyto columns to feature list
cyto_cols = [c for c in df.columns if any(x in c for x in ['+', '-', 't(', 'inv(', 'Complex', 'normal'])]
total_features.extend(cyto_cols)


# --- 2. Molecular Features (VAF Weighting) ---
print("Processing Molecular Features...")
# Clean Genes using parsing.py
maf_df['GENE_CLEAN'] = maf_df['GENE'].apply(parsing.parse_GENE)
maf_eval['GENE_CLEAN'] = maf_eval['GENE'].apply(parsing.parse_GENE)

# Pivot: ID x GENE (Value = Max VAF)
gene_train = maf_df.pivot_table(index='ID', columns='GENE_CLEAN', values='VAF', aggfunc='max', fill_value=0)
gene_test = maf_eval.pivot_table(index='ID', columns='GENE_CLEAN', values='VAF', aggfunc='max', fill_value=0)

# Prefix columns
gene_train.columns = [f"GENE_{c}" for c in gene_train.columns]
gene_test.columns = [f"GENE_{c}" for c in gene_test.columns]

# Align Test to Train
gene_test = gene_test.reindex(columns=gene_train.columns, fill_value=0)

# Merge
df = df.merge(gene_train, on='ID', how='left').fillna(0)
df_eval = df_eval.merge(gene_test, on='ID', how='left').fillna(0)
total_features.extend(gene_train.columns.tolist())

# One-Hot Encoding for Protein & Effect
for feature in ['EFFECT', 'PROTEIN_CHANGE']:
    col_name = 'temp'
    # Use parsing.py for PROTEIN_CHANGE
    if feature == 'PROTEIN_CHANGE':
        maf_df[col_name] = maf_df[feature].apply(parsing.parse_PROTEIN_CHANGE)
        maf_eval[col_name] = maf_eval[feature].apply(parsing.parse_PROTEIN_CHANGE)
    else:
        maf_df[col_name] = maf_df[feature]
        maf_eval[col_name] = maf_eval[feature]
        
    # Get Dummies
    dummies_train = pd.get_dummies(maf_df[col_name], prefix=feature)
    dummies_train['ID'] = maf_df['ID']
    agg_train = dummies_train.groupby('ID').max()
    
    dummies_test = pd.get_dummies(maf_eval[col_name], prefix=feature)
    dummies_test['ID'] = maf_eval['ID']
    agg_test = dummies_test.groupby('ID').max()
    
    # Align
    agg_test = agg_test.reindex(columns=agg_train.columns, fill_value=0)
    
    # Merge
    df = df.merge(agg_train, on='ID', how='left').fillna(0)
    df_eval = df_eval.merge(agg_test, on='ID', how='left').fillna(0)
    total_features.extend(agg_train.columns.tolist())


# --- 3. Clinical & Interactions ---
print("Processing Clinical Data...")
clinical_cols = ['BM_BLAST', 'WBC', 'ANC', 'PLT', 'HB', 'MONOCYTES']
for col in clinical_cols:
    df[col] = np.log1p(df[col])
    df_eval[col] = np.log1p(df_eval[col])

# Impute Missing Clinical
imputer = SimpleImputer(strategy='median')
df[clinical_cols] = imputer.fit_transform(df[clinical_cols])
df_eval[clinical_cols] = imputer.transform(df_eval[clinical_cols])
total_features.extend(clinical_cols)

# Add Interactions
def add_interactions(data):
    if 'GENE_NPM1' in data.columns and 'GENE_FLT3' in data.columns:
        data['INT_NPM1_pos_FLT3_neg'] = ((data['GENE_NPM1'] > 0) & (data['GENE_FLT3'] == 0)).astype(int)
    if 'GENE_TP53' in data.columns and 'Complex_Karyotype' in data.columns:
        data['INT_TP53_Complex'] = ((data['GENE_TP53'] > 0) & (data['Complex_Karyotype'] > 0)).astype(int)
    return data

df = add_interactions(df)
df_eval = add_interactions(df_eval)

# Add interactions to list
for feat in ['INT_NPM1_pos_FLT3_neg', 'INT_TP53_Complex']:
    if feat in df.columns:
        total_features.append(feat)

# De-fragment frames before training
df = df.copy()
df_eval = df_eval.copy()

# Final Feature List Cleanup
total_features = list(set([f for f in total_features if f in df.columns]))
print(f"Feature Engineering Complete. Total Features: {len(total_features)}")

Processing Cytogenetics...
Processing Molecular Features...
Processing Clinical Data...
Feature Engineering Complete. Total Features: 1773


## 4. Search for Gradient Boosting

### 4.1 Grid search

In [27]:
# Cell 4
# Prepare Data
X_full = df[total_features]
y_full = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', df)

# Split for validation
X_train, X_test, y_train, y_test = train_test_split(X_full, y_full, test_size=0.2, random_state=42)

# Define Model
gbsa = GradientBoostingSurvivalAnalysis(random_state=42, verbose=0, min_samples_leaf=10)

# Parameter Grid
param_grid = {
    'n_estimators': [500,1000,1500],
    'learning_rate': [0.01],
    'max_depth': [10,20,30],
    'min_samples_leaf': [5,10,20,30],
    'subsample': [0.1,0.5,1]  # Stochastic Gradient Boosting
}

print("Running Grid Search for GBSA...")
search = GridSearchCV(
    gbsa, 
    param_grid, 
    cv=3, 
    n_jobs=-1, 
    verbose=1
)
search.fit(X_train, y_train)

print("\nBest Parameters:", search.best_params_)
print("Best CV Score:", search.best_score_)

# Evaluate on local test set with tau=7
best_gbsa = search.best_estimator_
c_index_test = concordance_index_ipcw(y_train, y_test, best_gbsa.predict(X_test), tau=7)[0]
print(f"Test C-Index (tau=7): {c_index_test:.4f}")

Running Grid Search for GBSA...
Fitting 3 folds for each of 108 candidates, totalling 324 fits


KeyboardInterrupt: 

### 4.2 Single search

In [ ]:
# Cell 4
# Prepare Data
X_full = df[total_features]
y_full = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', df)

# Split for validation
X_train, X_test, y_train, y_test = train_test_split(X_full, y_full, test_size=0.2, random_state=42)

# Define Model
gbsa = GradientBoostingSurvivalAnalysis(max_depth=20,
                                        max_features='sqrt',
                                        n_estimators=1000,
                                        min_samples_leaf=10,
                                        subsample=0.1,
                                        learning_rate=0.01,
                                        random_state=42,
                                        verbose=1)

gbsa.fit(X_train, y_train)

# Evaluate on local test set with tau=7
c_index_test = concordance_index_ipcw(y_train, y_test, gbsa.predict(X_test), tau=1)[0]
print(f"Test C-Index (tau=7): {c_index_test:.4f}")

      Iter       Train Loss      OOB Improve   Remaining Time 
         1         589.1507           0.0000            1.71m
         2         549.8143         -63.0130            1.47m
         3         608.4796          89.1938            1.39m
         4         606.0320          -3.9677            1.36m
         5         619.0124          13.5415            1.33m
         6         654.3389          52.3900            1.31m
         7         670.8173          29.7832            1.30m
         8         651.2283         -25.8860            1.29m
         9         610.9334         -64.5024            1.29m
        10         593.0141         -15.9304            1.28m
        20         612.1378          20.4226            1.28m
        30         601.6643         -54.6101            1.26m
        40         600.3808         -22.8192            1.24m
        50         628.3372         -28.2499            1.21m
        60         613.2899          52.3667            1.19m
       

In [13]:
gbsa.fit(X_full, y_full)

# 2. Prepare Evaluation Data
X_eval_final = df_eval.reindex(columns=total_features, fill_value=0)

# 3. Predict
predictions = gbsa.predict(X_eval_final)

# 4. Save Submission
submission = pd.DataFrame({
    'ID': df_eval['ID'],
    'risk_score': predictions
})

filename = "submission/submission_gbsa.csv"
submission.to_csv(filename, index=False)
print(f"Saved {filename}")
submission.head()

      Iter       Train Loss      OOB Improve   Remaining Time 
         1         811.5646           2.4855            2.31m
         2         775.8809         -42.8908            2.11m
         3         816.8341          53.8072            2.00m
         4         731.9574        -114.5138            1.94m
         5         806.7225         111.7522            1.93m
         6         755.7378         -68.5833            1.91m
         7         764.8041          16.6160            1.90m
         8         876.7460         156.6721            1.89m
         9         833.1107         -51.3216            1.89m
        10         766.9977         -93.5952            1.89m
        11         746.6705         -23.1452            1.87m
        12         767.8677          30.1572            1.86m
        13         790.0661          32.7769            1.85m
        14         760.8973         -38.4647            1.84m
        15         826.0211         103.3063            1.83m
       

,ID,risk_score
0,KYW1,0.972107
1,KYW2,0.974540
2,KYW3,0.160973
3,KYW4,0.950650
4,KYW5,0.993136


## 5. Full Training & Submission

In [ ]:
# Cell 5
# 1. Retrain on Full Dataset
print("Retraining Best GBSA on Full Dataset...")
best_params = search.best_params_
final_model = GradientBoostingSurvivalAnalysis(**best_params, random_state=42)
final_model.fit(X_full, y_full)

# 2. Prepare Evaluation Data
X_eval_final = df_eval.reindex(columns=total_features, fill_value=0)

# 3. Predict
predictions = final_model.predict(X_eval_final)

# 4. Save Submission
submission = pd.DataFrame({
    'ID': df_eval['ID'],
    'risk_score': predictions
})

filename = "submission/submission_gbsa.csv"
submission.to_csv(filename, index=False)
print(f"Saved {filename}")
submission.head()